유효 컬럼: 픽업/드롭오프/패신저/딛스턴스/페어/팁/톨스/토탈/페이먼트

** 회귀 모델

데이터파일 :  taxi.csv

구현 모델 : 택시요금 예측 모델

In [ ]:
## 모듈로딩
import pandas as pd  # 데이터 모듈
import numpy as np

import torch         
# tensor 및 기본 함수 모듈   
import torch.nn as nn
# 인공신경망 관련 모듈
import torch.nn.functional as F
# 인공신경망 관련 함수
import torch.optim as optim
# 최적화 모듈

from sklearn.model_selection import train_test_split
# 학습용 데이터셋 관련 함수

from torchmetrics.regression import *
from torch.utils.data import Dataset, DataLoader

In [ ]:
dataDF = pd.DataFrame(pd.read_csv('../_data/taxis.csv', engine='python'))
taxDF = dataDF.copy()

In [ ]:
taxDF.info()

In [ ]:
taxDF = taxDF.dropna()

In [ ]:
for i in taxDF.columns:
    print(i, taxDF[i].unique().shape)

In [ ]:
for i in ['passengers','payment', 'pickup_borough', 'dropoff_borough']:
    print(i, taxDF[i].value_counts())

In [ ]:
taxDF[taxDF['pickup_borough']=='Bronx'].shape

In [ ]:
taxDF[taxDF['dropoff_borough']=='Bronx'].shape

In [ ]:
taxDF[(taxDF['pickup_borough']=='Bronx') | (taxDF['dropoff_borough']=='Bronx')].index

In [ ]:
taxDF = taxDF.drop(index=taxDF[(taxDF['pickup_borough']=='Bronx') | (taxDF['dropoff_borough']=='Bronx')].index)
taxDF = taxDF.drop(index= taxDF[taxDF['dropoff_borough']=='Staten Island'].index)

In [ ]:
taxDF.head()

In [ ]:
for i in ['passengers','payment', 'pickup_borough', 'dropoff_borough']:
    print(i, taxDF[i].value_counts())

 'passengers'
 ordinal로 처리 필요없음

 ['payment', 'pickup_borough', 'dropoff_borough']
 ohe로 처리하기

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

In [ ]:
taxDF = taxDF.reset_index(drop=True)

In [ ]:
taxDF.shape

In [ ]:
ohe = OneHotEncoder()
encoded_payment = ohe.fit_transform(taxDF[['payment']]).toarray()
taxDF= pd.concat((taxDF, pd.DataFrame(encoded_payment, columns=ohe.get_feature_names_out(['payment']))), axis=1)

ohe = OneHotEncoder()
encoded_payment = ohe.fit_transform(taxDF[['pickup_borough']]).toarray()
taxDF= pd.concat((taxDF, pd.DataFrame(encoded_payment, columns=ohe.get_feature_names_out(['pickup_borough']))), axis=1)

ohe = OneHotEncoder()
encoded_payment = ohe.fit_transform(taxDF[['dropoff_borough']]).toarray()
taxDF= pd.concat((taxDF, pd.DataFrame(encoded_payment, columns=ohe.get_feature_names_out(['dropoff_borough']))), axis=1)

In [ ]:
len('2019-03-23')

In [ ]:
for i in taxDF['pickup'].index:
    taxDF['pickup'].iloc[i] = int(taxDF['pickup'].iloc[i][11:13])
    taxDF['dropoff'].iloc[i] = int(taxDF['dropoff'].iloc[i][11:13])

In [ ]:
taxDF['pickup'] = taxDF['pickup'].astype('Int64')
taxDF['dropoff'] = taxDF['dropoff'].astype('Int64')

In [ ]:
taxDF['pickup'].value_counts()

In [ ]:
newDF = taxDF.drop(columns=['pickup_zone','dropoff_zone','pickup_borough','dropoff_borough','payment','color'])



In [ ]:
newDF.columns[newDF.columns != 'total']

In [ ]:
featureDF = newDF.loc[:, newDF.columns[newDF.columns != 'total']]
targetDF = newDF.loc[:,'total']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    featureDF, targetDF, train_size= 0.8, random_state=42)

print(f"X_train => {X_train.ndim}D {X_train.shape} / X_test => {X_test.ndim}D, {X_test.shape}")
print(f"y_train => {y_train.ndim}D {y_train.shape}, / y_test => {y_test.ndim}D, {y_test.shape}")

In [ ]:
## 데이터셋 클래스 생성

class TaxiDataSet(Dataset):
    #피쳐와 타겟 분리 및 전처리 진행
    def __init__(self, featureDF, targetDF):
        super().__init__()
        self.feature = featureDF
        self.target = targetDF
        self.rows = featureDF.shape[0]
        self.cols = featureDF.shape[1]
        
    # 데이터셋의 샘플 수 반환 메서드
    def __len__(self):
        return self.rows
        
    # DataLoader 에서 batch_size만큼 호출하는 메서드
    # 인덱스에 해당하는 피쳐와 타겟 반환, 단 Tensor 형태
    def __getitem__(self, index):
        arrFeature = self.feature.iloc[index].values
        print(arrFeature)
        arrTarget = self.target.iloc[index].reshape(-1,1)
        print(arrTarget)
        
        return torch.FloatTensor(arrFeature), torch.FloatTensor(arrTarget)

In [ ]:
dataset = TaxiDataSet(featureDF, targetDF)
dataset.feature.shape, dataset.target.shape

In [ ]:
## DataLoader 연동
loader = DataLoader(dataset, batch_size=1)
for feature, target in loader:
    print(feature, target)
    break
    


In [ ]:
trainDS = TaxiDataSet(X_train, y_train)
testDS  = TaxiDataSet(X_test, y_test)

In [ ]:

class Total_Model(nn.Module):
    # 인스턴스 초기화 및 생성 메서드
    def __init__(self):
        super().__init__() #부모(nn.Module)의 init 생성
        print('__init__()')
        self.layers = nn.Sequential(
                        nn.Linear(15, 400),
                        nn.ReLU(),
                        nn.Linear(400, 200),
                        nn.ReLU(),
                        nn.Linear(200, 100),
                        nn.ReLU(),
                        nn.Linear(100, 50),
                        nn.ReLU(),
                        nn.Linear(50, 1),
        )

    # 순전파 학습 메서드 
    def forward(self, data):
        return self.layers(data)
               

In [ ]:
model = Total_Model()

In [ ]:
for feature, target  in testDS:
    pre_ = model(feature)
    print(pre_, target)
    print(pre_.max(), pre_.argmax())
    break

In [ ]:
X_train.shape

In [ ]:
## [5-1] 학습 관련 설정들
EPOCHS = 30                                         # 학습용 DS를 처음부터 끝까지 1번 학습하는 것을 에포크
BATCH_SIZE = 4936//2                                      # DS를 학습량 만큼 나눈 사이즈
ITERATION = int(X_train.shape[0]/BATCH_SIZE)         # 학습용 DS이 분리된 수 => 1에포크에 W, b 업데이트 횟수
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"DEVICE : {DEVICE}, EPOCHS : {EPOCHS}, BATCH_SIZE : {BATCH_SIZE} ITERATION: {ITERATION}")
      

In [ ]:
# [5-2] 학습 관련 인스턴스들
TRAINDL   = DataLoader(trainDS, batch_size=BATCH_SIZE)  #학습용
TESTDL    = DataLoader(testDS,  batch_size=BATCH_SIZE)  #테스트용
LR      = 0.0001                                # Learning Rate
MODEL  = Total_Model()                           #학습 모델
OPTIMIZER = optim.Adam(MODEL.parameters(), lr=LR)      #최적화 즉, 경사하강법 알고리즘으로 W, b의 값 개신
LOSS_FN = nn.MSELoss()


In [ ]:
##- 학습 함수 --------------------------------------------
##- 학습용 데이터셋으로 모델 검증
##- -----------------------------------------------------
def training():
    # 학습 모드 설정
    model.train()

    E_LOSS = 0
    E_MSE = 0
    E_MAE = 0
    E_R2 = 0
    #for i in range(ITERATION): 
    for feature, target in TRAINDL:
        # 배치 크기만큼 feature, target 로딩
        # print(feature.shape, target.shape)
                                                                            
        # 가중치 기울기 0 초기화
        OPTIMIZER.zero_grad()

        # 학습 진행
        pre_y = MODEL(feature)

        # 손실 계산
        loss = LOSS_FN(pre_y, target.reshape(-1))
        
        # 역전파 진행
        loss.backward()

        # 가중치/절편 업데이트
        OPTIMIZER.step()
        
        # 정확도 계산
        MSE = MeanSquaredError()
        mse = MSE(pre_y, target.reshape(-1,1))
        MAE = MeanAbsoluteError()
        mae = MAE(pre_y, target.reshape(-1,1))
        R2 = R2Score()
        r2 = R2(pre_y, target.reshape(-1,1))
       
       
        E_R2 += r2.item()
        E_MAE += mae.item()
        E_MSE += mse.item()
        E_LOSS += loss.item()

    return E_LOSS/ITERATION, E_MSE/ITERATION, E_MAE/ITERATION, E_R2/ITERATION

In [ ]:
# 학습함수
def evaluate():
    # 에포크 단위로 검증 => 검증 모드
    MODEL.eval()
    
    # W, b가 업데이트 해제
    with torch.no_grad():
        # 방법1) 테스트용 DL 적용
        E_LOSS = 0
        E_MSE = 0
        E_MAE = 0
        E_R2 = 0
        CNT = 0 
        for feature, target in TESTDL:            
            # 검증진행
            pre_y= MODEL(feature)
            
            # 손실 계산
            loss = LOSS_FN(pre_y, target.reshape(-1).long())
            
            # 정확도 계산
            MSE = MeanSquaredError()
            mse = MSE(pre_y, target.reshape(-1,1))
            MAE = MeanAbsoluteError()
            mae = MAE(pre_y, target.reshape(-1,1))
            R2 = R2Score()
            r2 = R2(pre_y, target.reshape(-1,1))
        
        
            E_R2 += r2.item()
            E_MAE += mae.item()
            E_MSE += mse.item()
            E_LOSS += loss.item()
            E_LOSS += loss.item()
            CNT += 1
            return E_LOSS/ITERATION, E_MSE/ITERATION, E_MAE/ITERATION, E_R2/ITERATION
    
        # 방법2) 전테 테스트데이터
        # DF/SR => Tensor화
        # 검증용 데이터셋 => 텐서화 ndarray ==> tensor변환
        x = torch.FloatTensor(X_test.values) 
        y = torch.FloatTensor(y_test.values)
        
        # 검증진행
        pre_y= MODEL(x)
        
        # 손실 계산
        loss = LOSS_FN(pre_y, y.reshape(-1))

        # 정확도 계산
        MSE = MeanSquaredError()
        mse = MSE(pre_y, target.reshape(-1))
        MAE = MeanAbsoluteError()
        mae = MAE(pre_y, target.reshape(-1))
        R2 = R2Score()
        r2 = R2(pre_y, target.reshape(-1))
        
        return loss.item(), mse.item(), mae.item(), r2.item()

In [ ]:
evaluate()


In [ ]:
# 방법 2: time 모듈 사용
import time

# 현재 시간을 초 단위로 출력
print(time.time())

# 포맷팅된 시간 출력
print(time.strftime("%H:%M:%S"))

In [ ]:
HIST ={'Train':[], 'Valid':[]}   
# 에포크 단위 학습/검증 진행 
for epoch in range(EPOCHS):
    a = time.time()
    trainLoss, trainMse, trainMae, trainR2 = training()
    validLoss, validMse, validMae, validR2 = evaluate()

    HIST['Train'].append((trainLoss, trainMse, trainMae, trainR2 ))
    HIST['Valid'].append((validLoss, validMse, validMae, validR2))

    print(f'\nEPOCH[{epoch}/{EPOCHS}]----------------')
    print(f'- TRAIN_LOSS {trainLoss:.5f}    Mse {trainMse:.5f}   MAE {trainMae:.5f}  R2{trainR2:.5f}')
    print(f'- VALID_LOSS {validLoss:.5f}    Mse {validMse:.5f}   MAE {validMae:.5f}  R2{validR2:.5f}')
    
    print(time.time()-a)
print(time.strftime("%H:%M:%S"))

In [ ]:
# MODEL.eval()
xdata = range(100)
ydata1Loss = []
ydata1MSE = []
ydata1MAE = []
ydata1R2 = []

for loss, Mse, Mae, R2 in HIST['Train']:
    ydata1Loss.append(loss)
    ydata1MSE.append(Mse)
    ydata1MAE.append(Mae)
    ydata1R2.append(R2)

ydata2Loss = []
ydata2MSE = []
ydata2MAE = []
ydata2R2 = []

for loss, Mse, Mae, R2 in HIST['Valid']:
    ydata2Loss.append(loss)
    ydata2MSE.append(Mse)
    ydata2MAE.append(Mae)
    ydata2R2.append(R2)
    




In [ ]:
## 모델 성능 시각화 
import matplotlib.pyplot as plt

# plt.plot(HIST['Train'][0], 'b^--', label='TrainLOSS')
# plt.plot(HIST['Valid'][0], 'r^--', label='ValidLOSS')
# plt.plot(HIST['Train'][1], 'bo--', label='TrainACC')
# plt.plot(HIST['Valid'][1], 'ro--', label='ValidACC')

fig, ax = plt.subplots(1,4, figsize=(12,6))
ax = ax.flatten()
ax[0].plot(ydata1Loss, 'r^--', label='TrainLOSS')
ax[0].plot(ydata2Loss, 'b^--', label='ValidLOSS')

ax[1].plot(ydata1MSE, 'r^--', label='TrainMSE')
ax[1].plot(ydata2MSE, 'b^--', label='ValidMSE')

ax[2].plot(ydata1MAE, 'r^--', label='TrainMAE')
ax[2].plot(ydata2MAE, 'b^--', label='ValidMAE')

ax[3].plot(ydata1R2, 'r^--', label='TrainR2')
ax[3].plot(ydata1R2, 'b^--', label='ValidR2')

ax[0].grid();ax[1].grid();ax[2].grid();ax[3].grid()
ax[0].legend();ax[1].legend();ax[2].legend();ax[3].legend()

ax[0].set_xlabel('EPOCHS')
ax[1].set_xlabel('EPOCHS')

ax[0].set_ylabel('LOSS')
ax[1].set_ylabel('MSE')
ax[2].set_ylabel('MAE')
ax[3].set_ylabel('R2')
plt.tight_layout()
plt.show()